# Hướng 2, bước 3: train lại các lớp verb / noun / action trên đầu ra pooler (loss cân bằng lớp + head verb–noun)

Pooler của probe Meta được giữ nguyên; chỉ 3 lớp Linear được train lại trên **tập train** (`scripts/train_head.py`). Mỗi "recipe":

| recipe | ý nghĩa |
|---|---|
| `meta` | probe Meta, không train (baseline) |
| `focal` | train tiếp bằng sigmoid focal loss như Meta (đối chứng: chỉ "train thêm" thì được bao nhiêu) |
| `ce` | softmax cross-entropy thường |
| `bsm` | **Balanced Softmax**: CE trên `z + log prior` (mô hình học khi biết lớp nào phổ biến, lúc test bỏ prior đi) |
| `la` | logit-adjusted CE với `--la_tau` |
| `+cb` / `+sqrt` | lấy mẫu cân bằng theo lớp action (cRT) / theo căn bậc hai |
| `+compose` | **head action có cấu trúc**: `z_a + b_v·log_softmax(z_v)[verb(a)] + b_n·log_softmax(z_n)[noun(a)]`, b_v, b_n học được |
| `+random` | khởi tạo ngẫu nhiên thay vì từ probe Meta |

Mỗi recipe được báo cáo **thô** và **sau cách chấm lại post-hoc** (τ, β cross-fitted theo người tham gia của val, như
`posthoc_ek100.py`), để so sánh trực tiếp "cân bằng lúc train" với "chỉnh sau khi có kết quả" (32.7 → 39.3). Epoch được chọn trên
10% người tham gia của **train** được giữ lại; không có gì được tune trên val ngoài phần cross-fitting đó. 3 seed, mean ± std, khoảng
tin cậy bootstrap của (recipe − meta).

**Cài đặt:** GPU T4 (1 GPU là đủ; CPU cũng chạy được nhưng chậm), Internet **On**. *Add Input*: output của bước 2 (`feats_pooled`, `work`).

In [ ]:
ANCHORS = ["action_start", "official"]
RECIPES = "meta focal ce bsm la bsm+cb bsm+compose focal+compose"
SEEDS = "0 1 2"
EPOCHS = 10

In [ ]:
BRANCH = "direction2-balanced-head"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /tmp/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /tmp/vjepa2 && cd /tmp/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops av
import os
os.environ["VJEPA2_ROOT"] = "/tmp/vjepa2"
R = "/tmp/repo"
!git -C $R log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
# annotations (the full train csv: probe classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /tmp/ek100-ann
ANN = "/tmp/ek100-ann"

In [ ]:
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
import glob, os
# find the extraction output (a dataset, or the B2_Extract_Pooler notebook output attached via Add Input -> Notebooks)
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/work/train_clips.csv", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(os.path.dirname(p)), "feats_pooled"))]
if not cands:
    print("Input attached to this version (3 levels):")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 3:
            print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")
        if depth >= 3:
            dirs[:] = []
    raise SystemExit("No extraction output found (folders feats_pooled + work). Add Input -> Notebooks -> "
                     "B2_Extract_Pooler (the version that ran ~12 h), then Save Version again.")
W = cands[0]
FEATS = os.path.join(os.path.dirname(W), "feats_pooled")
print("features:", FEATS, "| clip index:", W)
# 1-minute self-test on fake data: catches environment problems before the real run
!python $R/scripts/train_head.py --synthetic /tmp/syn --out /tmp/syn/heads.json --recipes meta ce+random bsm+random bsm+compose \
    --seeds 0 --epochs 5 --bootstrap 20 | tail -6

In [ ]:
for anchor in ANCHORS:
    if not glob.glob(f"{FEATS}/train/{anchor}/shard*/chunk_*.npz"):
        print("no train features for", anchor); continue
    !python $R/scripts/train_head.py --work_dir $W --train_csv $ANN/EPIC_100_train.csv \
        --train_feats $FEATS/train/{anchor} --val_feats $FEATS/val/{anchor} --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt \
        --recipes $RECIPES --seeds $SEEDS --epochs $EPOCHS --feats_on_gpu \
        --out /kaggle/working/heads_{anchor}.json --logits_out /kaggle/working/head_logits

In [ ]:
# Gain by class-frequency group / prior in the logits / calibration for the best recipes (CPU, uses analyze_posthoc.py)
for anchor in ANCHORS:
    d = "/kaggle/working/head_logits"
    if os.path.isdir(d):
        args = " ".join(f"{anchor}_{r.replace('+', '_')}={d}/{r}/{anchor}" for r in ("meta", "bsm", "bsm+compose")
                        if os.path.isdir(f"{d}/{r}/{anchor}"))
        !python $R/scripts/analyze_posthoc.py --work_dir $W --train_csv $ANN/EPIC_100_train.csv --logits {args} \
            --out_dir /kaggle/working/analysis_{anchor} --no_figures | tail -30

Gửi lại bảng cuối của mỗi anchor (cell 5), cùng các file `heads_*.json`.